# 05 · ONNX export of the final classifier

**Goal:** one `classifier.onnx` file that the API can run on a plain CPU, without PyTorch.

The final model is EfficientNet-B3 on brain-cropped scans (`classifier_strong_crop.pt`, picked in `03_final_classifier.ipynb`). The ONNX graph has two outputs:

- `logits`: class scores, `[batch, 4]`
- `cams`: one heatmap per class, `[batch, 4, 12, 12]`. It equals Grad-CAM for this model head (checked in `04_gradcam.ipynb`), so the API draws heatmaps from ONNX alone.

Checks:
1. The numpy preprocessing (what the API will use) gives the same tensor as the training pipeline.
2. ONNX and PyTorch agree on every val image: same predictions, same errors, same heatmaps.
3. Speed of one image on CPU.

Runs locally (no GPU needed). **Inputs:** the dataset, `splits.csv` and `classifier_strong_crop.pt`.

In [ ]:
import json
import os
import time
from pathlib import Path

import albumentations as A
import cv2
import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
import timm
import torch
from albumentations.pytorch import ToTensorV2
from PIL import Image


def find_data_dir() -> Path:
    if "NEUROLENS_DATA" in os.environ:
        return Path(os.environ["NEUROLENS_DATA"])
    for p in Path("/kaggle/input").glob("**/Training"):
        return p.parent
    raise FileNotFoundError("Dataset not found. Attach masoudnickparvar/brain-tumor-mri-dataset.")


DATA_DIR = find_data_dir()
SPLITS_CSV = Path(os.environ.get("NEUROLENS_SPLITS", "../data/splits.csv"))
MODELS_DIR = Path(os.environ.get("NEUROLENS_MODELS", "../models"))
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CKPT = MODELS_DIR / "classifier_strong_crop.pt"
ONNX_PATH = MODELS_DIR / "classifier.onnx"
ARCH, IMG_SIZE, OPSET = "efficientnet_b3", 384, 18
CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
SEED = 42
torch.manual_seed(SEED)
print("data:", DATA_DIR, "| checkpoint:", CKPT)

## 1. Preprocessing in plain numpy

The API gets the same steps as training: crop to the head, resize to 384, normalize. Only numpy + OpenCV, so the server needs no PyTorch or Albumentations.

In [ ]:
def crop_brain(img: np.ndarray, margin: float = 0.03) -> np.ndarray:
    """Same crop as in 02_classifier: cut the slice to the head and pad it to a square."""
    gray = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), (5, 5), 0)
    mask = (gray > max(10, 0.08 * np.percentile(gray, 99.5))).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask)
    if n < 2:
        return img
    x, y, w, h, area = stats[1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])]
    if area < 0.05 * mask.size:
        return img
    m = int(margin * max(w, h))
    x0, y0 = max(0, x - m), max(0, y - m)
    x1, y1 = min(img.shape[1], x + w + m), min(img.shape[0], y + h + m)
    crop = img[y0:y1, x0:x1]
    side = max(crop.shape[:2])
    out = np.zeros((side, side, 3), dtype=img.dtype)
    oy, ox = (side - crop.shape[0]) // 2, (side - crop.shape[1]) // 2
    out[oy:oy + crop.shape[0], ox:ox + crop.shape[1]] = crop
    return out


def preprocess(img: np.ndarray) -> np.ndarray:
    """RGB uint8 image -> float32 [1, 3, 384, 384], ready for the ONNX model."""
    img = cv2.resize(crop_brain(img), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    x = (img.astype(np.float32) / 255.0 - MEAN) / STD
    return np.ascontiguousarray(x.transpose(2, 0, 1)[None])


def load_rgb(path: Path) -> np.ndarray:
    return np.array(Image.open(path).convert("RGB"))


df = pd.read_csv(SPLITS_CSV)
val_df = df[df.split == "val"].reset_index(drop=True)
y_val = val_df.label.map({c: i for i, c in enumerate(CLASSES)}).to_numpy()

# Training pipeline (Albumentations) vs the numpy version on a few val images
train_tf = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Normalize(mean=MEAN.tolist(), std=STD.tolist()), ToTensorV2()])
diffs = []
for p in val_df.path.sample(20, random_state=SEED):
    img = load_rgb(DATA_DIR / p)
    diffs.append(np.abs(train_tf(image=crop_brain(img))["image"].numpy() - preprocess(img)[0]).max())
print(f"max difference numpy vs training preprocessing: {max(diffs):.2e}")
assert max(diffs) < 1e-4

## 2. Export

A small wrapper returns the class scores and the per-class CAMs in one pass. The batch size stays dynamic (1 in the API, more for batch checks).

In [ ]:
class ClassifierWithCam(torch.nn.Module):
    def __init__(self, model: torch.nn.Module):
        super().__init__()
        self.model = model

    def forward(self, x):
        feats = self.model.forward_features(x)                                      # [B, C, 12, 12]
        logits = self.model.forward_head(feats)                                     # [B, 4]
        cams = torch.einsum("kc,bchw->bkhw", self.model.get_classifier().weight, feats)  # [B, 4, 12, 12]
        return logits, cams


model = timm.create_model(ARCH, pretrained=False, num_classes=len(CLASSES))
model.load_state_dict(torch.load(CKPT, map_location="cpu"))
wrapper = ClassifierWithCam(model).eval()

# Two example images: with a batch of 1 the exporter would fix the batch size to 1
example = torch.from_numpy(np.concatenate([preprocess(load_rgb(DATA_DIR / p)) for p in val_df.path[:2]]))
torch.onnx.export(
    wrapper, (example,), ONNX_PATH,
    input_names=["image"], output_names=["logits", "cams"], opset_version=OPSET,
    dynamic_shapes={"x": {0: torch.export.Dim("batch", min=1, max=64)}},
    external_data=False,  # one self-contained file
)
onnx.checker.check_model(onnx.load(ONNX_PATH))
size_mb = ONNX_PATH.stat().st_size / 1e6
print(f"saved {ONNX_PATH} ({size_mb:.1f} MB)")

## 3. ONNX vs PyTorch on every val image

In [ ]:
sess = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
print("ONNX inputs:", [(i.name, i.shape) for i in sess.get_inputs()], "| outputs:", [(o.name, o.shape) for o in sess.get_outputs()])

BATCH = 16
pt_logits, ox_logits, cam_corr, cam_diff = [], [], [], []
for s in range(0, len(val_df), BATCH):
    x = np.concatenate([preprocess(load_rgb(DATA_DIR / p)) for p in val_df.path[s:s + BATCH]])
    with torch.no_grad():
        lt, ct = (t.numpy() for t in wrapper(torch.from_numpy(x)))
    lo, co = sess.run(None, {"image": x})
    pt_logits.append(lt)
    ox_logits.append(lo)
    k = lt.argmax(1)
    for b in range(len(x)):  # compare the heatmap of the predicted class
        a, c = ct[b, k[b]].ravel(), co[b, k[b]].ravel()
        cam_corr.append(np.corrcoef(a, c)[0, 1])
        cam_diff.append(np.abs(a - c).max() / (np.abs(a).max() + 1e-8))
pt_logits, ox_logits = np.concatenate(pt_logits), np.concatenate(ox_logits)

pt_pred, ox_pred = pt_logits.argmax(1), ox_logits.argmax(1)
check = dict(
    n_val=len(val_df),
    max_logit_diff=float(np.abs(pt_logits - ox_logits).max()),
    same_prediction=int((pt_pred == ox_pred).sum()),
    pytorch_val_acc=round(float((pt_pred == y_val).mean()), 4), pytorch_val_errors=int((pt_pred != y_val).sum()),
    onnx_val_acc=round(float((ox_pred == y_val).mean()), 4), onnx_val_errors=int((ox_pred != y_val).sum()),
    min_cam_correlation=float(np.min(cam_corr)), max_cam_rel_diff=float(np.max(cam_diff)),
)
print(json.dumps(check, indent=2))
assert check["same_prediction"] == len(val_df) and check["max_logit_diff"] < 1e-3

## 4. Speed on CPU (one image)

Median of repeated runs, batch 1, like one upload in the API. Preprocessing (crop + resize) is timed separately.

In [ ]:
def median_ms(fn, runs: int = 30, warmup: int = 5) -> float:
    for _ in range(warmup):
        fn()
    times = []
    for _ in range(runs):
        t = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t) * 1000)
    return float(np.median(times))


img = load_rgb(DATA_DIR / val_df.path[0])
x1 = preprocess(img)
torch.set_num_threads(os.cpu_count())
with torch.no_grad():
    speed = dict(
        preprocess_ms=round(median_ms(lambda: preprocess(img)), 1),
        onnx_cpu_ms=round(median_ms(lambda: sess.run(None, {"image": x1})), 1),
        pytorch_cpu_ms=round(median_ms(lambda: wrapper(torch.from_numpy(x1))), 1),
        cpu_threads=os.cpu_count(),
    )
print(json.dumps(speed, indent=2))

## 5. Save the report

In [ ]:
report = dict(model=ARCH, checkpoint=CKPT.name, onnx_file=ONNX_PATH.name, onnx_size_mb=round(size_mb, 1), opset=OPSET,
              input="image [batch, 3, 384, 384], RGB, brain crop + ImageNet normalize",
              outputs=dict(logits="[batch, 4]", cams="[batch, 4, 12, 12]"), classes=CLASSES,
              check=check, speed=speed)
(OUT_DIR / "onnx_report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))